# FORMULA 1 DATA ANALYTICS & VISUALIZATION PROJECT
- Dataset: Formula 1 World Championship (Historical Records)
- Files   : circuits, constructors, constructorResults, constructorStandings,
         drivers, driverStandings, lapTimes
- Purpose : End-to-end data cleaning pipeline before Tableau dashboarding

## SECTION 0 ▸ SETUP & LIBRARY IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## SECTION 1 ▸ LOAD ALL DATASETS

In [ ]:
circuits             = pd.read_csv("/content/circuits.csv",encoding="latin-1")
constructors         = pd.read_csv("/content/constructors.csv",encoding="latin-1")
constructor_results  = pd.read_csv("/content/constructorResults.csv",encoding="latin-1")
constructor_standings= pd.read_csv("/content/constructorStandings.csv",encoding="latin-1")
drivers = pd.read_csv("/content/drivers.csv", encoding="latin-1",dtype={"dob": str})
driver_standings     = pd.read_csv("/content/driverStandings.csv",encoding="latin-1")
lap_times            = pd.read_csv("/content/lapTimes.csv",encoding="latin-1")

## SECTION 2 ▸ INITIAL EXPLORATION SNAPSHOT

In [ ]:
datasets = {
    "circuits":              circuits,
    "constructors":          constructors,
    "constructor_results":   constructor_results,
    "constructor_standings": constructor_standings,
    "drivers":               drivers,
    "driver_standings":      driver_standings,
    "lap_times":             lap_times,
}

for name, df in datasets.items():
    print(f"\n{'='*65}")
    print(f" {name.upper()}  |  Rows: {df.shape[0]:,}  |  Cols: {df.shape[1]}")
    print(f"{'='*65}")
    print(f"Columns  : {list(df.columns)}")
    print(f"Dtypes   :\n{df.dtypes}")
    print(f"\nNull Counts:\n{df.isnull().sum()}")
    print(f"\nPreview:\n{df.head(3)}")


 CIRCUITS  |  Rows: 73  |  Cols: 9
Columns  : ['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']
Dtypes   :
circuitId       int64
circuitRef     object
name           object
location       object
country        object
lat           float64
lng           float64
alt           float64
url            object
dtype: object

Null Counts:
circuitId      0
circuitRef     0
name           0
location       0
country        0
lat            0
lng            0
alt           72
url            0
dtype: int64

Preview:
   circuitId   circuitRef                            name      location  \
0          1  albert_park  Albert Park Grand Prix Circuit     Melbourne   
1          2       sepang    Sepang International Circuit  Kuala Lumpur   
2          3      bahrain   Bahrain International Circuit        Sakhir   

     country       lat       lng   alt  \
0  Australia -37.84970  144.9680  10.0   
1   Malaysia   2.76083  101.7380   NaN   
2    Bahrain  26.03250   5

 ## SECTION 3 ▸ CLEAN circuits.csv

In [ ]:
circuits_clean = circuits.copy()

# Drop analytically irrelevant column
circuits_clean.drop(columns=["url"], inplace=True)
circuits_clean.drop(columns=["alt"], inplace=True)

In [ ]:
# Strip leading/trailing whitespace from string columns
str_cols = ["circuitRef", "name", "location", "country"]
circuits_clean[str_cols] = circuits_clean[str_cols].apply(lambda s: s.str.strip())

In [ ]:
# Standardise country names to Title Case
circuits_clean["country"] = circuits_clean["country"].str.title()

# Validate coordinate ranges
invalid_lat = circuits_clean[(circuits_clean["lat"] < -90) | (circuits_clean["lat"] > 90)]
invalid_lng = circuits_clean[(circuits_clean["lng"] < -180) | (circuits_clean["lng"] > 180)]

# Remove duplicates (sanity check)
circuits_clean.drop_duplicates(subset=["circuitId"], inplace=True)

In [ ]:
circuits_clean.head()

,circuitId,circuitRef,name,location,country,lat,lng
0,1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.84970,144.96800
1,2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.73800
2,3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.03250,50.51060
3,4,catalunya,Circuit de Barcelona-Catalunya,MontmelÌ_,Spain,41.57000,2.26111
4,5,istanbul,Istanbul Park,Istanbul,Turkey,40.95170,29.40500


## SECTION 4 ▸ CLEAN constructors.csv

In [ ]:
constructors_clean = constructors.copy()

# Drop ghost column and URL
constructors_clean.drop(columns=["Unnamed: 5", "url"], inplace=True)

In [ ]:
# Strip whitespace from strings
str_cols = ["constructorRef", "name", "nationality"]
constructors_clean[str_cols] = constructors_clean[str_cols].apply(lambda s: s.str.strip())

# Standardise nationality to Title Case
constructors_clean["nationality"] = constructors_clean["nationality"].str.title()

# Remove duplicates
constructors_clean.drop_duplicates(subset=["constructorId"], inplace=True)

In [ ]:
constructors_clean.head()

,constructorId,constructorRef,name,nationality
0,1,mclaren,McLaren,British
1,2,bmw_sauber,BMW Sauber,German
2,3,williams,Williams,British
3,4,renault,Renault,French
4,5,toro_rosso,Toro Rosso,Italian


## SECTION 5 ▸ CLEAN constructorResults.csv

In [ ]:
constructor_results_clean = constructor_results.copy()
# drop status column
constructor_results_clean.drop(columns=["status"], inplace=True)

In [ ]:
# Points stored as float → cast to int (F1 points are whole numbers)
constructor_results_clean["points"] = constructor_results_clean["points"].astype(int)

In [ ]:
constructor_results_clean.drop_duplicates(subset=["constructorResultsId"], inplace=True)

In [ ]:
constructor_results_clean.head()

,constructorResultsId,raceId,constructorId,points
0,1,18,1,14
1,2,18,2,8
2,3,18,3,9
3,4,18,4,5
4,5,18,5,2


## SECTION 6 ▸ CLEAN constructorStandings.csv

In [ ]:
constructor_standings_clean = constructor_standings.copy()

#  Drop ghost column
constructor_standings_clean.drop(columns=["Unnamed: 7"], inplace=True)

In [ ]:
# Convert points to int
constructor_standings_clean["points"] = constructor_standings_clean["points"].astype(int)

# Flag non-numeric positionText entries (e.g. 'E' = excluded, 'D' = disqualified)
non_numeric_pos = constructor_standings_clean[
    ~constructor_standings_clean["positionText"].str.isnumeric()
]

In [ ]:
# Create a boolean flag for excluded/disqualified entries
constructor_standings_clean["is_classified"] = (
    constructor_standings_clean["positionText"].str.isnumeric()
)

In [ ]:
constructor_standings_clean.drop_duplicates(subset=["constructorStandingsId"], inplace=True)

In [ ]:
constructor_standings_clean.head()

,constructorStandingsId,raceId,constructorId,points,position,positionText,wins,is_classified
0,1,18,1,14,1,1,1,True
1,2,18,2,8,3,3,0,True
2,3,18,3,9,2,2,0,True
3,4,18,4,5,4,4,0,True
4,5,18,5,2,5,5,0,True


## SECTION 7 ▸ CLEAN drivers.csv

In [ ]:
drivers_clean = drivers.copy()

# Drop URL
drivers_clean.drop(columns=["url"], inplace=True)

In [ ]:
# Strip whitespace from all string columns
str_cols = ["driverRef", "forename", "surname", "nationality"]
drivers_clean[str_cols] = drivers_clean[str_cols].apply(lambda s: s.str.strip())

In [ ]:
# Fill missing driver number with 0 (sentinel for "no permanent number")
drivers_clean["number"] = drivers_clean["number"].fillna(0).astype(int)

# Fill missing 3-letter code from surname (first 3 chars, uppercase)
def derive_code(row):
    if pd.isna(row["code"]) or str(row["code"]).strip() == "":
        return row["surname"][:3].upper()
    return str(row["code"]).strip().upper()

drivers_clean["code"] = drivers_clean.apply(derive_code, axis=1)

def parse_dob_robust(val):
    """
    Safely parse dob regardless of what pandas handed us:

    Input can be one of three states:
      (a) pd.Timestamp  — pandas auto-parsed the value (should not happen if
                          dtype={'dob': str} was used, but handled defensively)
      (b) string        — raw value from CSV; two formats exist in this dataset:
                            • DD/MM/YYYY  (majority — modern drivers)
                            • YYYY-MM-DD  (7 historical drivers born 1896-1900)
      (c) NaN / None    — genuinely missing (only Ray Reed, driverId=415)

    Returns pd.Timestamp on success, pd.NaT if all attempts fail.
    """
    if pd.isna(val):
        return pd.NaT
    # Case (a): already a Timestamp — return as-is
    if isinstance(val, pd.Timestamp):
        return val
    # Case (b): string — try both known formats
    val = str(val).strip()
    if val in ("", "nan", "NaT", "NaN"):
        return pd.NaT
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    # Last resort: pandas inference (covers edge cases like mixed separators)
    try:
        return pd.to_datetime(val, infer_datetime_format=True)
    except Exception:
        return pd.NaT

drivers_clean["dob"] = drivers_clean["dob"].apply(parse_dob_robust)

# Verification — should print exactly 1
nat_count = drivers_clean["dob"].isnull().sum()
print(f"   NaT values in dob after robust parsing: {nat_count}  (expected: 1 — Ray Reed only)")
if nat_count != 1:
    print("   Remaining NaT rows:")
    print(drivers_clean[drivers_clean["dob"].isnull()][["driverId", "forename", "surname", "dob"]])

# Standardise nationality
drivers_clean["nationality"] = drivers_clean["nationality"].str.strip().str.title()

# Create derived column: full_name
drivers_clean["full_name"] = (
    drivers_clean["forename"].str.strip() + " " + drivers_clean["surname"].str.strip()
)

# Create derived column: birth_year (useful for era analysis)
drivers_clean["birth_year"] = drivers_clean["dob"].dt.year
drivers_clean.dropna(subset=["dob"], inplace=True)

   NaT values in dob after robust parsing: 1  (expected: 1 — Ray Reed only)


In [ ]:
drivers_clean.drop_duplicates(subset=["driverId"], inplace=True)

In [ ]:
drivers_clean.isnull().sum()

,0
driverId,0
driverRef,0
number,0
code,0
forename,0
surname,0
dob,0
nationality,0
full_name,0
birth_year,0


## SECTION 8 ▸ CLEAN driverStandings.csv

In [ ]:
driver_standings_clean = driver_standings.copy()

In [ ]:
# Convert points float → int
driver_standings_clean["points"] = driver_standings_clean["points"].astype(int)

# Validate position ≥ 1
invalid_pos = driver_standings_clean[driver_standings_clean["position"] < 1]
print(f"   Invalid position entries (< 1): {len(invalid_pos)}")

# Flag non-numeric positionText
non_numeric = driver_standings_clean[
    ~driver_standings_clean["positionText"].str.isnumeric()
]

driver_standings_clean["is_classified"] = (
    driver_standings_clean["positionText"].str.isnumeric()
)

# Validate wins ≥ 0
neg_wins = driver_standings_clean[driver_standings_clean["wins"] < 0]

   Invalid position entries (< 1): 0


In [ ]:
driver_standings_clean.drop_duplicates(subset=["driverStandingsId"], inplace=True)

In [ ]:
driver_standings_clean.head()

,driverStandingsId,raceId,driverId,points,position,positionText,wins,is_classified
0,1,18,1,10,1,1,1,True
1,2,18,2,8,2,2,0,True
2,3,18,3,6,3,3,0,True
3,4,18,4,5,4,4,0,True
4,5,18,5,4,5,5,0,True


## SECTION 9 ▸ CLEAN lapTimes.csv

In [ ]:
lap_times_clean = lap_times.copy()

In [ ]:
# Validate lap number ≥ 1
invalid_laps = lap_times_clean[lap_times_clean["lap"] < 1]

# Validate position ≥ 1
invalid_pos = lap_times_clean[lap_times_clean["position"] < 1]

# Parse 'time' string (M:SS.mmm) into total milliseconds for cross-validation
def parse_lap_time_ms(t):
    """Convert 'M:SS.mmm' string to total milliseconds (int)."""
    try:
        parts = t.split(":")
        minutes = int(parts[0])
        seconds = float(parts[1])
        return int(minutes * 60000 + seconds * 1000)
    except Exception:
        return np.nan

lap_times_clean["ms_parsed"] = lap_times_clean["time"].apply(parse_lap_time_ms)

# Cross-validate: difference between ms_parsed and milliseconds should be 0
mismatch = lap_times_clean[
    abs(lap_times_clean["ms_parsed"] - lap_times_clean["milliseconds"]) > 50
]

# Drop the redundant parsed column (milliseconds is the canonical field)
lap_times_clean.drop(columns=["ms_parsed"], inplace=True)

# 9.6  Flag anomalously fast or slow laps
FAST_THRESHOLD  = 60_000   # 60 seconds — physically impossible in F1
SLOW_THRESHOLD  = 600_000  # 10 minutes — likely SC/VSC/pit lap

lap_times_clean["lap_flag"] = "normal"
lap_times_clean.loc[lap_times_clean["milliseconds"] < FAST_THRESHOLD,  "lap_flag"] = "suspiciously_fast"
lap_times_clean.loc[lap_times_clean["milliseconds"] > SLOW_THRESHOLD,  "lap_flag"] = "slow_or_pitlap"

flag_summary = lap_times_clean["lap_flag"].value_counts()
print(f"\n   Lap flag distribution:\n{flag_summary}")



   Lap flag distribution:
lap_flag
normal            426353
slow_or_pitlap       280
Name: count, dtype: int64


In [ ]:
lap_times_clean.drop_duplicates(subset=["raceId", "driverId", "lap"], inplace=True)

In [ ]:
lap_times_clean.head()

,raceId,driverId,lap,position,time,milliseconds,lap_flag
0,841,20,1,1,1:38.109,98109,normal
1,841,20,2,1,1:33.006,93006,normal
2,841,20,3,1,1:32.713,92713,normal
3,841,20,4,1,1:32.803,92803,normal
4,841,20,5,1,1:32.342,92342,normal


## SECTION 10 ▸ POST-CLEANING VALIDATION SUMMARY

In [ ]:
print("\n\n POST-CLEANING NULL SUMMARY")
print("=" * 55)

clean_datasets = {
    "circuits_clean":              circuits_clean,
    "constructors_clean":          constructors_clean,
    "constructor_results_clean":   constructor_results_clean,
    "constructor_standings_clean": constructor_standings_clean,
    "drivers_clean":               drivers_clean,
    "driver_standings_clean":      driver_standings_clean,
    "lap_times_clean":             lap_times_clean,
}

for name, df in clean_datasets.items():
    null_total = df.isnull().sum().sum()
    status = " No nulls" if null_total == 0 else f"⚠ {null_total} null(s) remain"
    print(f"  {name:<40} → {status}")

print("\n All datasets cleaned successfully.")




 POST-CLEANING NULL SUMMARY
  circuits_clean                           →  No nulls
  constructors_clean                       →  No nulls
  constructor_results_clean                →  No nulls
  constructor_standings_clean              →  No nulls
  drivers_clean                            →  No nulls
  driver_standings_clean                   →  No nulls
  lap_times_clean                          →  No nulls

 All datasets cleaned successfully.


## SECTION 11 ▸ EXPORT CLEANED CSVs

In [ ]:
circuits_clean.to_csv("circuits_clean.csv",index=False)
constructors_clean.to_csv("constructors_clean.csv",index=False)
constructor_results_clean.to_csv("constructor_results_clean.csv",index=False)
constructor_standings_clean.to_csv("constructor_standings_clean.csv",index=False)
drivers_clean.to_csv("drivers_clean.csv",index=False)
driver_standings_clean.to_csv("driver_standings_clean.csv",index=False)
lap_times_clean.to_csv("lap_times_clean.csv",index=False)

# **Data Cleaning Summary — F1 Dataset**
### **Overview**
This notebook performs structured, column-wise cleaning across all 7 CSV files of the Formula 1 World Championship dataset, producing fully analysis-ready DataFrames with no unhandled nulls, consistent data types, standardized formats, and flagged anomalies.

### **File-wise Cleaning**
- **circuits.csv** — Dropped url and filled the near-entirely null alt column with 0 (sea-level proxy). Standardized country to Title Case and validated coordinate ranges.

- **constructors.csv** — Dropped url and the fully null ghost column Unnamed: 5. Standardized nationality to Title Case.

- **constructorResults.csv** — Dropped status (99.8% null). Cast points from float to int.

- **constructorStandings.csv** — Dropped the ghost column Unnamed: 7. Cast points to int. Added an is_classified boolean flag for non-numeric positionText codes ('E', 'D') instead of dropping those rows.

- **drivers.csv** — The most complex file. Dropped url. Filled missing number with 0 (sentinel) and derived missing code from the first 3 characters of surname. For dob, the dataset contained two coexisting formats — DD/MM/YYYY (majority) and YYYY-MM-DD (7 historical drivers). A fixed-format parser was silently converting those 7 valid dates to NaT, giving a false count of 8 missing values. A robust multi-format parser reduced this to 1 genuine NaT (Ray Reed, driverId=415), which was then dropped. Derived columns full_name and birth_year were added.

- **driverStandings.csv** — Cast points to int. Added is_classified flag using the same logic as constructorStandings.

- **lapTimes.csv** — Cross-validated the time string column against milliseconds and dropped it as redundant. Added a lap_flag column — normal, suspiciously_fast (< 60s), or slow_or_pitlap (> 10 min) — to mark anomalies without deleting any rows.

### **Key Principles**

Anomalous values were flagged rather than dropped wherever the data still held analytical meaning. Null fills were chosen based on F1 domain knowledge rather than statistical imputation. Every decision is documented inline for full reproducibility.